## Imports

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import classification_report, roc_auc_score

## Charger les données

In [4]:
import os

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
file_path = os.path.join(BASE_DIR, "data", "clean_data.csv")

df = pd.read_csv(file_path)
df.head()

,age,tenure_months,monthly_logins,weekly_active_days,avg_session_time,features_used,usage_growth_rate,last_login_days_ago,monthly_fee,total_revenue,...,contract_type_Yearly,payment_method_Card,payment_method_PayPal,discount_applied_Yes,price_increase_last_3m_Yes,complaint_type_No_Complaint,complaint_type_Service,complaint_type_Technical,survey_response_Satisfied,survey_response_Unsatisfied
0,68,22,26,7,11.762372,5,0.06,7,30,660,...,False,False,True,True,False,False,True,False,True,False
1,57,9,7,5,26.846390,1,-0.28,2,30,270,...,False,True,False,False,True,False,False,False,False,False
2,24,58,19,5,23.380065,6,0.13,23,20,1160,...,True,True,False,False,False,False,True,False,False,False
3,49,19,34,7,24.243136,2,-0.17,24,30,570,...,True,False,False,True,False,False,False,True,False,False
4,65,52,20,6,18.872323,2,-0.16,2,50,2600,...,False,False,True,False,False,False,False,True,False,True


## séparer X/y

In [ ]:
X = df.drop("churn", axis=1)
y = df["churn"]

### train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
feature_names = X_train.columns.tolist() 
print (feature_names)

### Scaling

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Modèle 

### Logistic Regression (baseline)

In [ ]:
log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)
y_proba_log = log_model.predict_proba(X_test)[:, 1]

print("=== Logistic Regression (baseline) ===")
print(classification_report(y_test, y_pred_log, target_names=['Non-Churn','Churn']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_log):.4f}")

### Random forest

In [ ]:
rf = RandomForestClassifier(random_state=42)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

# À ajouter dans Cell 14 (après rf)
print("=== Random Forest (baseline) ===")
print(classification_report(y_test, y_pred_rf, target_names=['Non-Churn','Churn']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_rf):.4f}")

### Gradient Boosting

In [ ]:
gb = GradientBoostingClassifier(random_state=42)

gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)
y_proba_gb = gb.predict_proba(X_test)[:, 1]

print("=== Gradient Boosting (baseline) ===")
print(classification_report(y_test, y_pred_gb, target_names=['Non-Churn','Churn']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_gb):.4f}")

### MLP 

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    max_iter=300,
    random_state=42
)

mlp.fit(X_train, y_train)

y_pred_mlp = mlp.predict(X_test)
y_proba_mlp = mlp.predict_proba(X_test)[:, 1]

print("=== MLP — Deep Learning (baseline) ===")
print(classification_report(y_test, y_pred_mlp, target_names=['Non-Churn','Churn']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_mlp):.4f}")

## Evaluation

### Fonction

In [ ]:
def evaluate(y_test, y_pred, y_proba):
    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
        "Pred_0 (Non-Churn)": (y_pred == 0).sum(),
        "Pred_1 (Churn)": (y_pred == 1).sum()
    }

### Tableau comparatif

In [ ]:
results = pd.DataFrame([
    evaluate(y_test, y_pred_log, y_proba_log),
    evaluate(y_test, y_pred_rf, y_proba_rf),
    evaluate(y_test, y_pred_gb, y_proba_gb),
    evaluate(y_test, y_pred_mlp, y_proba_mlp)
], index=["Logistic Regression", "Random Forest", "Gradient Boosting", "MLP"])

print(results)

In [ ]:
results.sort_values(by="F1", ascending=False)

In [ ]:
# Logistic Regression avec class_weight
lr_balanced = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_balanced.fit(X_train, y_train)
y_pred_lr = lr_balanced.predict(X_test)
print("=== Logistic Regression (balanced) ===")
print(classification_report(y_test, y_pred_lr))
print(f"ROC-AUC: {roc_auc_score(y_test, lr_balanced.predict_proba(X_test)[:,1]):.4f}")

## ref

In [ ]:
# Logistic Regression avec class_weight
lr_balanced = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_balanced.fit(X_train, y_train)
y_pred_lr = lr_balanced.predict(X_test)
print("=== Logistic Regression (balanced) ===")
print(classification_report(y_test, y_pred_lr))
print(f"ROC-AUC: {roc_auc_score(y_test, lr_balanced.predict_proba(X_test)[:,1]):.4f}")

In [ ]:
rf_balanced = RandomForestClassifier(
    class_weight='balanced_subsample',  # ← plus stable que 'balanced' pour RF
    n_estimators=200,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf_balanced.fit(X_train, y_train)
y_pred_rf = rf_balanced.predict(X_test)
y_proba_rf = rf_balanced.predict_proba(X_test)[:,1]

print("=== RF (balanced_subsample) ===")
print(classification_report(y_test, y_pred_rf, zero_division=0))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_rf):.4f}")

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from collections import Counter

# Vérifier la distribution avant
print("Avant SMOTE:", Counter(y_train))

# Appliquer SMOTE
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Après SMOTE:", Counter(y_train_sm))
# → Les deux classes seront équilibrées

# Entraîner Gradient Boosting sur données SMOTE (meilleur modèle de base)
from sklearn.ensemble import GradientBoostingClassifier

gb_smote = GradientBoostingClassifier(random_state=42)
gb_smote.fit(X_train_sm, y_train_sm)
y_pred_gb_sm = gb_smote.predict(X_test)

print("\n=== Gradient Boosting + SMOTE ===")
print(classification_report(y_test, y_pred_gb_sm))
print(f"ROC-AUC: {roc_auc_score(y_test, gb_smote.predict_proba(X_test)[:,1]):.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import precision_recall_curve, f1_score, precision_score, recall_score, classification_report, roc_auc_score

# Récupère les probabilités du GB original
y_proba_gb = gb_smote.predict_proba(X_test)[:,1]

# --- Tester plusieurs seuils ---
thresholds = np.arange(0.1, 0.9, 0.05)
recalls, precisions, f1s = [], [], []

for thresh in thresholds:
    y_pred_t = (y_proba_gb >= thresh).astype(int)
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_t, zero_division=0))
    f1s.append(f1_score(y_test, y_pred_t, zero_division=0))

# Trouver le seuil optimal (max F1)
best_thresh_idx = np.argmax(f1s)
best_thresh = thresholds[best_thresh_idx]
print(f"Seuil optimal : {best_thresh:.2f}")
print(f"  → Precision : {precisions[best_thresh_idx]:.3f}")
print(f"  → Recall    : {recalls[best_thresh_idx]:.3f}")
print(f"  → F1        : {f1s[best_thresh_idx]:.3f}")

# --- Visualisation ---
plt.figure(figsize=(10, 4))
plt.plot(thresholds, precisions, label='Precision', marker='o')
plt.plot(thresholds, recalls, label='Recall', marker='s')
plt.plot(thresholds, f1s, label='F1-score', marker='^', linewidth=2)
plt.axvline(best_thresh, color='red', linestyle='--', label=f'Seuil optimal = {best_thresh:.2f}')
plt.xlabel('Seuil de décision')
plt.ylabel('Score')
plt.title('Precision / Recall / F1 selon le seuil')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('threshold_analysis.png', dpi=150)
plt.show()

# --- Prédiction finale avec le seuil optimal ---
y_pred_optimal = (y_proba_gb >= best_thresh).astype(int)
print("\n=== GB + SMOTE avec seuil optimal ===")
print(classification_report(y_test, y_pred_optimal,
                             target_names=['Non-Churn', 'Churn']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_gb):.4f}")

In [ ]:
# XGBoost
from xgboost import XGBClassifier
from collections import Counter
from sklearn.metrics import classification_report, roc_auc_score, f1_score
import numpy as np

# Calcul automatique du ratio déséquilibre
counter = Counter(y_train)
scale = counter[0] / counter[1]
print(f"scale_pos_weight = {scale:.2f}")  # → ~8.79

# Modèle XGBoost
xgb_model = XGBClassifier(
    scale_pos_weight=scale,
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    random_state=42
)

xgb_model.fit(X_train, y_train)  # ← données originales (pas SMOTE), scale_pos_weight s'en occupe

y_proba_xgb = xgb_model.predict_proba(X_test)[:,1]

# --- Trouver le seuil optimal pour XGBoost ---
thresholds = np.arange(0.1, 0.9, 0.05)
f1s_xgb = []

for thresh in thresholds:
    y_pred_t = (y_proba_xgb >= thresh).astype(int)
    f1s_xgb.append(f1_score(y_test, y_pred_t, zero_division=0))

best_thresh_xgb = thresholds[np.argmax(f1s_xgb)]
y_pred_xgb = (y_proba_xgb >= best_thresh_xgb).astype(int)

print(f"\n=== XGBoost (seuil optimal={best_thresh_xgb:.2f}) ===")
print(classification_report(y_test, y_pred_xgb,
                             target_names=['Non-Churn', 'Churn']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_xgb):.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (roc_curve, auc, precision_recall_curve,
                              confusion_matrix, ConfusionMatrixDisplay)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Données de tous les modèles ---
models_proba = {
    'GB + SMOTE':  gb_smote.predict_proba(X_test)[:,1],
    'XGBoost':     y_proba_xgb
}

# --- 1. Courbe ROC ---
for name, proba in models_proba.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc(fpr,tpr):.3f})', linewidth=2)
axes[0].plot([0,1],[0,1],'k--', label='Aléatoire')
axes[0].set_title('Courbe ROC', fontsize=13)
axes[0].set_xlabel('Taux Faux Positifs')
axes[0].set_ylabel('Taux Vrais Positifs')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- 2. Courbe Precision-Recall ---
for name, proba in models_proba.items():
    p, r, _ = precision_recall_curve(y_test, proba)
    axes[1].plot(r, p, label=name, linewidth=2)
axes[1].axhline(y=0.10, color='gray', linestyle='--', label='Baseline (10%)')
axes[1].set_title('Courbe Precision-Recall', fontsize=13)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# --- 3. Matrice de confusion XGBoost ---
cm = confusion_matrix(y_test, y_pred_xgb)
disp = ConfusionMatrixDisplay(cm, display_labels=['Non-Churn', 'Churn'])
disp.plot(ax=axes[2], colorbar=False, cmap='Blues')
axes[2].set_title(f'Matrice de Confusion — XGBoost\n(seuil={best_thresh_xgb:.2f})', fontsize=13)

plt.suptitle('Évaluation Finale des Modèles — Churn Detection', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('courbes_finales.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd

# Feature importance
features_names = feature_names
importances = xgb_model.feature_importances_

fi_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False).reset_index(drop=True)

# Affichage top 15
plt.figure(figsize=(10, 6))
bars = plt.barh(fi_df['Feature'][:15][::-1], 
                fi_df['Importance'][:15][::-1], 
                color='steelblue', edgecolor='white')
plt.title('Top 15 Features — XGBoost Churn Prediction', fontsize=13, fontweight='bold')
plt.xlabel('Importance')
plt.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

print("\nTop 10 features :")
print(fi_df.head(10).to_string(index=False))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (precision_score, recall_score, 
                              f1_score, accuracy_score, roc_auc_score)

# ============================================================
# TABLEAU COMPARATIF FINAL — tous les modèles testés
# ============================================================

# Prédictions avec seuil par défaut (0.5) pour les modèles originaux
# (remplace par tes variables originales)
resultats = [
    {
        'Modèle':            'Logistic Regression',
        'Accuracy':          0.8965,
        'Precision (Churn)': 0.29,
        'Recall (Churn)':    0.01,
        'F1 (Churn)':        0.02,
        'ROC-AUC':           0.723,
        'Seuil':             0.50
    },
    {
        'Modèle':            'Random Forest',
        'Accuracy':          0.8985,
        'Precision (Churn)': 0.67,
        'Recall (Churn)':    0.01,
        'F1 (Churn)':        0.02,
        'ROC-AUC':           0.800,
        'Seuil':             0.50
    },
    {
        'Modèle':            'Gradient Boosting',
        'Accuracy':          0.8945,
        'Precision (Churn)': 0.42,
        'Recall (Churn)':    0.09,
        'F1 (Churn)':        0.15,
        'ROC-AUC':           0.805,
        'Seuil':             0.50
    },
    {
        'Modèle':            'GB + SMOTE (seuil=0.20)',
        'Accuracy':          0.75,
        'Precision (Churn)': 0.25,
        'Recall (Churn)':    0.76,
        'F1 (Churn)':        0.38,
        'ROC-AUC':           0.792,
        'Seuil':             0.20
    },
    {
        'Modèle':            'XGBoost ✅ FINAL',
        'Accuracy':          0.74,
        'Precision (Churn)': 0.26,
        'Recall (Churn)':    0.80,
        'F1 (Churn)':        0.39,
        'ROC-AUC':           0.795,
        'Seuil':             0.40
    }
]

df_results = pd.DataFrame(resultats)

# Affichage formaté
print("=" * 80)
print("TABLEAU COMPARATIF FINAL — CHURN PREDICTION")
print("=" * 80)
print(df_results.to_string(index=False))
print("=" * 80)
print("→ Modèle retenu : XGBoost (seuil=0.40)")
print("→ Justification  : Meilleur recall churn (0.80) + ROC-AUC (0.795)")

# ============================================================
# VISUALISATION DU TABLEAU
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models = df_results['Modèle']
x = range(len(models))

# Graphe 1 : Recall et F1 par modèle
axes[0].bar([i - 0.2 for i in x], df_results['Recall (Churn)'], 
            width=0.4, label='Recall Churn', color='steelblue')
axes[0].bar([i + 0.2 for i in x], df_results['F1 (Churn)'], 
            width=0.4, label='F1 Churn', color='orange')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=20, ha='right', fontsize=8)
axes[0].set_title('Recall & F1 (classe Churn) par modèle')
axes[0].set_ylabel('Score')
axes[0].legend()
axes[0].grid(True, axis='y', alpha=0.3)
axes[0].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Seuil cible')

# Graphe 2 : ROC-AUC par modèle
colors = ['gray', 'gray', 'gray', 'gray', 'green']
axes[1].bar(x, df_results['ROC-AUC'], color=colors, edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, rotation=20, ha='right', fontsize=8)
axes[1].set_title('ROC-AUC par modèle')
axes[1].set_ylabel('ROC-AUC')
axes[1].set_ylim(0.6, 0.85)
axes[1].grid(True, axis='y', alpha=0.3)

plt.suptitle('Comparaison Finale des Modèles — Churn Detection', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('comparaison_finale.png', dpi=150)
plt.show()

In [ ]:
# Trouver un churner BIEN détecté (proba élevée)
import numpy as np

# Prendre le churner avec la plus haute probabilité prédite
churner_indices = np.where(y_test.values == 1)[0]
best_churner_idx = churner_indices[np.argmax(y_proba_xgb[churner_indices])]

print(f"Client analysé : index {best_churner_idx}")
print(f"Probabilité de churn : {y_proba_xgb[best_churner_idx]:.2%}")
print(f"Prédiction : {'Churn ✅' if y_pred_xgb[best_churner_idx]==1 else 'Non-Churn ❌'}")

shap.force_plot(
    explainer.expected_value,
    shap_values[best_churner_idx],
    X_test[best_churner_idx],
    feature_names=feature_names,
    matplotlib=True,
    show=False
)
plt.savefig("shap_force_plot.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import pickle, json

# Sauvegarder XGBoost + scaler + features
with open('xgb_model.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('feature_names.json', 'w') as f:
    json.dump(feature_names, f)

# Vérifier les valeurs min/max pour les sliders Streamlit
import pandas as pd
df_original = pd.read_csv("data/clean_data.csv")
print(df_original.describe().round(1).to_string())